# Cost functions

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/alx87grd/minilink/blob/main/examples/tutorial/core/04_costs.ipynb)

A `CostFunction` defines the running cost density $g(x, u, t)$ and terminal cost $h(x, t)$ of the continuous-time objective

$$J = \int_{t_0}^{t_0 + t_f} e^{-\rho(t - t_0)}\, g(x(t), u(t), t)\, dt + e^{-\rho t_f}\, h(x(t_0 + t_f), t_0 + t_f),$$

while a `StepCost` (`SampledCost`) defines the per-tick stage cost $c(x_k, u_k, k)$ and terminal cost $h(x_N, N)$ of a discrete-time problem.

In [ ]:
# Local conda: minilink already installed. Colab: clone + path + meshcat.
import sys

if "google.colab" in sys.modules:
    get_ipython().run_line_magic("matplotlib", "inline")
    get_ipython().system("git clone https://github.com/alx87grd/minilink")
    sys.path.insert(0, "/content/minilink")
    get_ipython().system("pip install -q meshcat")

## 1. `CostFunction` and `QuadraticCost`: public API

| `CostFunction` Method / Property | Arguments | Output type & shape | Meaning |
| --- | --- | --- | --- |
| `cost.params` | — | `dict[str, Any]` | Cost parameters (`{"Q", "R", "S", "x_bar", "u_bar"}` on `QuadraticCost`) |
| `cost.g(x, u, t=0.0, params=None)` | `x: (n,)`, `u: (m,)`, `t: float = 0.0`, `params: dict \| None = None` | `float` (scalar) | Running cost density $g(x, u, t)$ |
| `cost.h(x, t=0.0, params=None)` | `x: (n,)`, `t: float = 0.0`, `params: dict \| None = None` | `float` (scalar) | Terminal cost $h(x, t)$ |
| `cost.integrate(traj, ...)` | `traj: Trajectory`, `*, discount_rate=0.0`, `params=None`, `t0=0.0` | `(cost_rate: array (N,), cost: array (N,))` | Discounted integrand $e^{-\rho(t_k - t_0)} g_k$ and cumulative trapezoidal integral $\int_{t_0}^{t_k} e^{-\rho(\tau - t_0)} g\, d\tau$ |
| `cost.g_max(X, U, params=None)` | `X: BoxSet`, `U: BoxSet`, `params=None` | `float` | Maximum (exact on `QuadraticCost` / `TimeCost`, sampled otherwise) of $g$ over $X \times U$ |
| `cost.h_max(X, params=None)` | `X: BoxSet`, `params=None` | `float` | Maximum of $h$ over the box $X$ |
| `c1 + c2` / `SumCost.of(*costs, **named)` | `CostFunction`s | `SumCost` | Sum of cost functions with nested `.params` keyed by term ID |
| `w * cost` (`ScaledCost(cost, w)`) | `w: float`, `cost: CostFunction` | `ScaledCost` | Cost scaled by scalar weight $w$, with `.params = {"weight": w, "cost": ...}` |
| `QuadraticCost.from_system(sys, ...)` | `sys: System`, `*, Q=None`, `R=None`, `S=None`, `x_bar=None`, `u_bar=None` | `QuadraticCost` | Build $(x - \bar x)^\top Q (x - \bar x) + (u - \bar u)^\top R (u - \bar u)$ and $(x - \bar x)^\top S (x - \bar x)$ from system dimensions and nominal point |

In [ ]:
import numpy as np

from minilink import Pendulum, QuadraticCost

plant = Pendulum()
cost = QuadraticCost.from_system(
    plant,
    Q=np.diag([10.0, 1.0]),
    R=0.1 * np.eye(1),
    S=5.0 * np.eye(2),
    x_bar=np.array([np.pi, 0.0]),
)
cost

In [ ]:
x = np.array([0.5, 0.0])
u = np.array([1.0])

g_val = cost.g(x, u, t=0.0)
h_val = cost.h(x, t=5.0)
g_val, h_val

In [ ]:
# Exact maxima of g and h over the plant's state and input boxes
cost.g_max(plant.state.box, plant.input_box.box), cost.h_max(plant.state.box)

## 2. Scoring and plotting a `Trajectory` with a cost

| Trajectory / System Cost Helper | Arguments | Output type & shape | Meaning |
| --- | --- | --- | --- |
| `traj.with_cost(cost, ...)` | `cost: CostFunction`, `*, discount_rate=0.0`, `params=None`, `t0=0.0` | `Trajectory` | Attach `"cost_rate"` and `"cost"` signals (each `(1, N)`) to `traj` |
| `sys.compute_cost(cost, ...)` | `cost: CostFunction`, `*, of: System \| None = None`, `traj: Trajectory \| None = None` | `float` | Total cost $J = \int g\, dt + h(x(t_N), t_N)$ of `traj` (or of subsystem `of` inside a diagram) |
| `sys.plot_cost(cost, ...)` | `cost: CostFunction`, `*, of: System \| None = None`, `traj: Trajectory \| None = None` | `PlotResult` | Plot `"cost_rate"` and cumulative `"cost"` over time |

In [ ]:
plant.x0 = np.array([1.0, 0.0])
traj = plant.compute_trajectory(tf=5.0, verbose=False)

cost_rate, running = cost.integrate(traj, discount_rate=0.1)
cost_rate.shape, running.shape, running[-1]

In [ ]:
scored_traj = traj.with_cost(cost)
scored_traj.signal_names, plant.compute_cost(cost, traj=traj)

In [ ]:
plant.plot_cost(cost, traj=traj)

## 3. Cost algebra (`SumCost`, `ScaledCost`), soft constraints, and `TimeCost`

Costs add with `+` (`SumCost`) and scale with `*` (`ScaledCost`). Use `SumCost.of(tracking=..., wall=...)` to give explicit names to the terms in `total.terms` and `total.params`.

`TimeCost(x_bar, radius=1e-3)` (or `TimeCost.from_system(sys, x_bar=..., radius=1e-3)`) is the minimum-time running cost: $g(x, u) = 0$ when $\|x - \bar x\|_2 < \varepsilon$ and $1$ otherwise, with $h(x) = 0$.

In [ ]:
from minilink import BoxSet, TimeCost

wall = BoxSet([-2.0, -3.0], [2.0, 3.0]).as_cost()
total = cost + 50.0 * wall
total

In [ ]:
named_total = type(total).of(tracking=cost, barrier=50.0 * wall)
list(named_total.terms), list(named_total.params), named_total.g(np.array([2.5, 0.0]), u)

In [ ]:
min_time = TimeCost.from_system(plant, x_bar=np.array([np.pi, 0.0]), radius=0.05)
min_time.g(np.array([0.0, 0.0]), u), min_time.g(np.array([np.pi, 0.0]), u), min_time.g_max(plant.state.box, plant.input_box.box)

### Writing a custom `CostFunction`

Subclass `CostFunction` and implement `g(self, x, u, t=0.0, params=None)` and `h(self, x, t=0.0, params=None)` using `array_module(x)` so the cost traces under both NumPy and JAX:

In [ ]:
import jax.numpy as jnp

from minilink import CostFunction
from minilink.core.backends import array_module


class SwingUpCost(CostFunction):
    def __init__(self):
        self.params = {"w_dtheta": 0.1, "w_u": 0.001}

    def g(self, x, u, t=0.0, params=None):
        params = self.params if params is None else params
        xp = array_module(x)
        e = xp.arctan2(xp.sin(x[0] - xp.pi), xp.cos(x[0] - xp.pi))
        return e**2 + params["w_dtheta"] * x[1] ** 2 + params["w_u"] * u[0] ** 2

    def h(self, x, t=0.0, params=None):
        return 0.0


swing_up = SwingUpCost()
swing_up.g(np.array([-3.1, 0.0]), u), swing_up.g(jnp.array([3.1, 0.0]), jnp.array([1.0]))

## 4. `StepCost` and `SampledCost`: costs on a clock

A `StepCost` defines a per-tick stage cost $c(x_k, u_k, k)$ and terminal cost $h(x_N, N)$ for the discrete-time objective

$$J = \sum_{k=0}^{N-1} \gamma^k\, c(x_k, u_k, k) + \gamma^N\, h(x_N, N).$$

`SampledCost(cost, plant, dt, *, solver="rk4", substeps=1, discount_rate=0.0, t0=0.0)` integrates a continuous `CostFunction` alongside the plant over one sampling period $\Delta t$:

| `SampledCost` Method | Arguments | Output type & shape | Meaning |
| --- | --- | --- | --- |
| `sampled.c(x, u, k=0, params=None)` | `x: (n,)`, `u: (m,)`, `k: int = 0`, `params=None` | `float` | Integrated stage cost $c_k = \int_{t_k}^{t_k + \Delta t} e^{-\rho(\tau - t_k)} g(x(\tau), u, \tau)\, d\tau$ |
| `sampled.h(x, k=0, params=None)` | `x: (n,)`, `k: int = 0`, `params=None` | `float` | Terminal cost $h(x, t_k)$ at $t_k = t_0 + k\,\Delta t$ |
| `sampled.stage_at(x, u, t_k, ...)` | `x: (n,)`, `u: (m,)`, `t_k: float`, `params=None`, `plant_params=None`, `discount_rate=None` | `(x_next: array (n,), c_k: float)` | Advance plant state and integrate stage cost in one fused step |

In [ ]:
from minilink.core import SampledCost

sampled_cost = SampledCost(cost, plant, dt=0.05, solver="rk4", substeps=1)
c_0 = sampled_cost.c(x, u, k=0)
h_0 = sampled_cost.h(x, k=0)
x_next, c_fused = sampled_cost.stage_at(x, u, t_k=0.0)

c_0, h_0, x_next, c_fused

Next, [`05_problem`](05_problem.ipynb): `PlanningProblem`, `StepPlanningProblem`, and `StepFace`.